# Model Pipeline Benchmark

Clean notebook for:
1. Optional training
2. Validation
3. ONNX export (FP32 + FP16)
4. Latency/FPS benchmark
5. Final comparison table


In [1]:
from pathlib import Path
import shutil
import sys
from typing import Any

import pandas as pd
import yaml
from ultralytics import YOLO

import onnxruntime as  ort

# Ensure `src` is importable even when notebook is launched from /notebooks
cwd = Path.cwd().resolve()
root_candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next(
    (p for p in root_candidates if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Could not locate project root from cwd: {cwd}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.training.benchmark_utils import (
    benchmark_onnx,
    benchmark_pt,
    build_tiled_dataset,
    detect_project_root,
    get_split_images,
    model_size_mb,
    parse_val_metrics,
    preprocess_for_onnx,
    rel_path,
    resolve_existing_weights,
    resolve_model_source,
    write_runtime_dataset_yaml,
)


In [2]:
# ==============================
# Global settings
# ==============================

# Pipeline switches
RUN_TRAIN = False
RUN_TILE = False  # Offline tiling: split-wise tiling before training/validation
RUN_VALIDATE = True
RUN_EXPORT = True
RUN_BENCHMARK = True
SAVE_REPORTS = True

# Benchmark settings
BENCHMARK_WARMUP = 10
BENCHMARK_RUNS = 80
MAX_BENCHMARK_IMAGES = None  # None -> use full split


PROJECT_ROOT = detect_project_root(Path.cwd())
TRAIN_CONFIG_PATH = PROJECT_ROOT / "configs" / "train_config.yaml"

with TRAIN_CONFIG_PATH.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f) or {}

train_cfg = cfg.get("train", {})
eval_cfg = cfg.get("evaluate", {})
export_cfg = cfg.get("export", {})

data_prep_cfg = cfg.get("data_preparation", {})
tile_cfg = data_prep_cfg.get("tiling", {}) if isinstance(data_prep_cfg.get("tiling", {}), dict) else {}

TILE_ROOT = (PROJECT_ROOT / data_prep_cfg.get("tile_root", "data/processed/TILED")).resolve()
tile_size = tile_cfg.get("tile_size", [640, 384])
if isinstance(tile_size, (list, tuple)) and len(tile_size) == 2:
    TILE_WIDTH = int(tile_size[0])
    TILE_HEIGHT = int(tile_size[1])
else:
    TILE_WIDTH = 640
    TILE_HEIGHT = 384
TILE_OVERLAP = float(tile_cfg.get("overlap", 0.25))
TILE_KEEP_RATIO = float(tile_cfg.get("keep_ratio", 0.5))
TILE_MIN_BOX_PX = int(tile_cfg.get("min_box_px", 4))
TILE_EMPTY_TO_POS_RATIO = float(tile_cfg.get("empty_to_positive_ratio", 1.0))
TILE_SEED = int(tile_cfg.get("seed", 42))

# Devices are taken from config
TRAIN_DEVICE = str(train_cfg.get("device", "cpu"))
EVAL_DEVICE = str(eval_cfg.get("device", TRAIN_DEVICE))
EXPORT_DEVICE = str(export_cfg.get("device", TRAIN_DEVICE))
BENCHMARK_DEVICE = EVAL_DEVICE

# Core params from config
EVAL_SPLIT = str(eval_cfg.get("split", "test"))
IMGSZ = int(eval_cfg.get("imgsz", train_cfg.get("imgsz", 640)))
BATCH = int(eval_cfg.get("batch", train_cfg.get("batch", 16)))

# Paths from config
weights_path = (PROJECT_ROOT / eval_cfg.get("weights", "models/weights/best.pt")).resolve()
dataset_yaml = (PROJECT_ROOT / eval_cfg.get("data", train_cfg.get("data", "configs/dataset.yaml"))).resolve()

export_dir = (PROJECT_ROOT / export_cfg.get("output_dir", "models/onnx")).resolve()
export_dir.mkdir(parents=True, exist_ok=True)
onnx_path = export_dir / "model.onnx"
onnx_fp16_path = export_dir / "model.fp16.onnx"

reports_dir = (PROJECT_ROOT / "reports").resolve()
reports_dir.mkdir(parents=True, exist_ok=True)
dataset_yaml_runtime = reports_dir / "dataset_abs.yaml"

print("project_root:", PROJECT_ROOT)
print("train_config:", TRAIN_CONFIG_PATH)
print("weights:", weights_path)
print("dataset_yaml:", dataset_yaml)
print("devices -> train/eval/export/bench:", TRAIN_DEVICE, EVAL_DEVICE, EXPORT_DEVICE, BENCHMARK_DEVICE)
print("run flags -> train/tile/validate/export/benchmark:", RUN_TRAIN, RUN_TILE, RUN_VALIDATE, RUN_EXPORT, RUN_BENCHMARK)
print("tiling -> root/size/overlap/keep/min_px/empty_ratio:", TILE_ROOT, (TILE_WIDTH, TILE_HEIGHT), TILE_OVERLAP, TILE_KEEP_RATIO, TILE_MIN_BOX_PX, TILE_EMPTY_TO_POS_RATIO)


project_root: C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline
train_config: C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\configs\train_config.yaml
weights: C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\models\weights\best26n_2_7394f.pt
dataset_yaml: C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\data\processed\YOLO\dataset.yaml
devices -> train/eval/export/bench: cuda cpu cuda cpu
run flags -> train/tile/validate/export/benchmark: False False True True True
tiling -> root/size/overlap/keep/min_px/empty_ratio: C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\data\processed\TILED (640, 384) 0.25 0.4 4 1.0


In [3]:
# Helper functions moved to src/training/benchmark_utils.py
# This notebook now keeps only pipeline steps and high-level config.


In [4]:
# Step 1: optional offline tiling (split-wise)

working_dataset_yaml = dataset_yaml

if RUN_TILE:
    tiled_dataset_yaml = build_tiled_dataset(
        source_dataset_yaml=dataset_yaml,
        tile_root=TILE_ROOT,
        tile_w=TILE_WIDTH,
        tile_h=TILE_HEIGHT,
        overlap=TILE_OVERLAP,
        keep_ratio=TILE_KEEP_RATIO,
        min_box_px=TILE_MIN_BOX_PX,
        empty_to_pos_ratio=TILE_EMPTY_TO_POS_RATIO,
        seed=TILE_SEED,
        project_root=PROJECT_ROOT,
    )
    working_dataset_yaml = tiled_dataset_yaml
    print("RUN_TILE=True -> using tiled dataset:", working_dataset_yaml)
else:
    print("RUN_TILE=False -> using original dataset:", working_dataset_yaml)


RUN_TILE=False -> using original dataset: C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\data\processed\YOLO\dataset.yaml


In [5]:
# Step 2: prepare runtime dataset config and benchmark tensors

dataset_yaml_runtime = write_runtime_dataset_yaml(working_dataset_yaml, dataset_yaml_runtime, project_root=PROJECT_ROOT)
sample_images = get_split_images(
    dataset_yaml_runtime,
    split_name=EVAL_SPLIT,
    project_root=PROJECT_ROOT,
    max_images=MAX_BENCHMARK_IMAGES,
)
if not sample_images:
    raise RuntimeError(f"No images found in split '{EVAL_SPLIT}'")

input_tensors = [preprocess_for_onnx(p, IMGSZ) for p in sample_images]

print("dataset_yaml_runtime:", dataset_yaml_runtime)
print("split:", EVAL_SPLIT)
print("images used for benchmark:", len(sample_images))
print("imgsz:", IMGSZ)


dataset_yaml_runtime: C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\reports\dataset_abs.yaml
split: test
images used for benchmark: 740
imgsz: 640


In [6]:
# Step 3: optional training

if RUN_TRAIN:
    train_args = dict(train_cfg)
    train_args["data"] = str(dataset_yaml_runtime)

    # Keep training runs inside project directory
    if "project" in train_args:
        train_args["project"] = str((PROJECT_ROOT / str(train_args["project"])).resolve())

    model_id = train_args.pop("model", "yolo11n.pt")
    model_source = resolve_model_source(str(model_id), project_root=PROJECT_ROOT)

    try:
        trainer = YOLO(model_source, task="detect")
        _ = trainer.train(**train_args)
        print("Training finished.")
    except TypeError as exc:
        msg = str(exc)
        if "SPPF.__init__" in msg:
            raise RuntimeError(
                "Model checkpoint is incompatible with current Ultralytics version. "
                f"model={model_source}. "
                "Use a supported model (e.g., yolo11n.pt) or install the Ultralytics version that produced this checkpoint."
            ) from exc
        raise
else:
    print("RUN_TRAIN=False -> training skipped")



RUN_TRAIN=False -> training skipped


In [7]:
# Step 4: validation (.pt)

weights_path = resolve_existing_weights(weights_path, project_root=PROJECT_ROOT)

pt_metrics = {}
pt_model = YOLO(str(weights_path), task="detect")

if RUN_VALIDATE:
    pt_val = pt_model.val(
        data=str(dataset_yaml_runtime),
        split=EVAL_SPLIT,
        imgsz=IMGSZ,
        batch=BATCH,
        device=EVAL_DEVICE,
        verbose=False,
    )
    pt_metrics = parse_val_metrics(pt_val)

print("pt_metrics:", pt_metrics)


[WARN] Weights not found: C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\models\weights\best26n_2_7394f.pt
[WARN] Using fallback weights: C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\models\weights\best26n_2_7394.pt
Ultralytics 8.4.24  Python-3.12.3 torch-2.7.1+cu128 CPU (Intel Core(TM) i5-7500 3.40GHz)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 667.7273.0 MB/s, size: 66.8 KB)
val: Scanning C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\data\processed\YOLO\labels\test.cache... 740 images, 92 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 740/740  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 1.4it/s 32.8s0.8ss
                   all        740       1196      0.634      0.474      0.506      0.262
Speed: 0.8ms preprocess, 37.4ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to C:\Users\sa

In [8]:
# Step 5: export ONNX (FP32 + FP16)

if RUN_EXPORT:
    exported = pt_model.export(
        format="onnx",
        imgsz=int(export_cfg.get("imgsz", IMGSZ)),
        dynamic=bool(export_cfg.get("dynamic", False)),
        simplify=bool(export_cfg.get("simplify", True)),
        half=False,
        device=EXPORT_DEVICE,
    )
    exported_path = Path(str(exported)).resolve()
    if exported_path != onnx_path:
        shutil.copy2(exported_path, onnx_path)

    try:
        exported_fp16 = pt_model.export(
            format="onnx",
            imgsz=int(export_cfg.get("imgsz", IMGSZ)),
            dynamic=bool(export_cfg.get("dynamic", False)),
            simplify=bool(export_cfg.get("simplify", True)),
            half=True,
            device=EXPORT_DEVICE,
        )
        exported_fp16_path = Path(str(exported_fp16)).resolve()
        if exported_fp16_path != onnx_fp16_path:
            shutil.copy2(exported_fp16_path, onnx_fp16_path)
    except Exception as exc:
        print("FP16 export failed:", repr(exc))

print("onnx_path:", onnx_path, "exists:", onnx_path.exists())
print("onnx_fp16_path:", onnx_fp16_path, "exists:", onnx_fp16_path.exists())


Ultralytics 8.4.24  Python-3.12.3 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce GTX 1080, 8192MiB)

PyTorch: starting from 'C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\weights\best26n_2_7394.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.1 MB)

ONNX: starting export with onnx 1.17.0 opset 17...


C:\Users\sanek\AppData\Roaming\Python\Python312\site-packages\torch\onnx\symbolic_opset9.py:5350: UserWarning: Exporting aten::index operator of advanced indexing in opset 17 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  warnings.warn(


ONNX: slimming with onnxslim 0.1.89...
ONNX: export success  2.4s, saved as 'C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\weights\best26n_2_7394.onnx' (9.4 MB)

Export complete (3.1s)
Results saved to C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\weights
Predict:         yolo predict task=detect model=C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\weights\best26n_2_7394.onnx imgsz=640 
Validate:        yolo val task=detect model=C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\weights\best26n_2_7394.onnx imgsz=640 data=C:\Users\sanek\OneDrive\ \cv_drone_pipeline\reports\dataset_abs.yaml  
Visualize:       https://netron.app
Ultralytics 8.4.24  Python-3.12.3 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce GTX 1080, 8192MiB)

PyTorch: starting from 'C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\weights\best26n_2_7394.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.1 MB)

ONNX: starting export with onnx 1.17.0 opset 17...
ONNX: slimming with on

In [9]:
# Step 6: validation (ONNX formats)

onnx_metrics = {}
onnx_fp16_metrics = {}

if RUN_VALIDATE and onnx_path.exists():
    try:
        onnx_model = YOLO(str(onnx_path), task="detect")
        onnx_val = onnx_model.val(
            data=str(dataset_yaml_runtime),
            split=EVAL_SPLIT,
            imgsz=IMGSZ,
            batch=BATCH,
            device=EVAL_DEVICE,
            verbose=False,
        )
        onnx_metrics = parse_val_metrics(onnx_val)
    except Exception as exc:
        print("ONNX validation failed:", repr(exc))

if RUN_VALIDATE and onnx_fp16_path.exists():
    try:
        onnx_fp16_model = YOLO(str(onnx_fp16_path), task="detect")
        onnx_fp16_val = onnx_fp16_model.val(
            data=str(dataset_yaml_runtime),
            split=EVAL_SPLIT,
            imgsz=IMGSZ,
            batch=BATCH,
            device=EVAL_DEVICE,
            verbose=False,
        )
        onnx_fp16_metrics = parse_val_metrics(onnx_fp16_val)
    except Exception as exc:
        print("ONNX FP16 validation failed:", repr(exc))

print("onnx_metrics:", onnx_metrics)
print("onnx_fp16_metrics:", onnx_fp16_metrics)


Ultralytics 8.4.24  Python-3.12.3 torch-2.7.1+cu128 CPU (Intel Core(TM) i5-7500 3.40GHz)
Loading C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\onnx\model.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.24.4 with CPUExecutionProvider
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access  (ping: 0.00.0 ms, read: 427.7205.7 MB/s, size: 83.1 KB)
val: Scanning C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\data\processed\YOLO\labels\test.cache... 740 images, 92 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 740/740  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 740/740 28.4it/s 26.0s<0.1s
                   all        740       1196      0.639      0.457      0.497      0.261
Speed: 0.7ms preprocess, 27.1ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to C:\Users\sanek\OneDrive\ \cv_drone_pipeline\notebooks\runs\detect\val25
Ultralytics 8.4.24  Python-3.12.3 torch-2.7.1+cu

In [10]:
# Step 7: benchmark + final comparison table

comparison_df = pd.DataFrame()

if RUN_BENCHMARK:
    rows: list[dict[str, Any]] = []

    pt_bench = benchmark_pt(
        weights=weights_path,
        tensors=input_tensors,
        device_name=BENCHMARK_DEVICE,
        warmup=BENCHMARK_WARMUP,
        runs=BENCHMARK_RUNS,
    )
    rows.append(
        {
            "model_format": ".pt",
            "path": rel_path(weights_path, project_root=PROJECT_ROOT),
            "size_mb": round(model_size_mb(weights_path), 2),
            "precision": round(float(pt_metrics.get("precision", float("nan"))), 4),
            "recall": round(float(pt_metrics.get("recall", float("nan"))), 4),
            "map50": round(float(pt_metrics.get("map50", float("nan"))), 4),
            "map50_95": round(float(pt_metrics.get("map50_95", float("nan"))), 4),
            "latency_ms": round(float(pt_bench["latency_ms"]), 3),
            "fps": round(float(pt_bench["fps"]), 2),
            "runtime": pt_bench["runtime"],
            "input_dtype": pt_bench["input_dtype"],
        }
    )

    if onnx_path.exists():
        onnx_bench = benchmark_onnx(
            model_path=onnx_path,
            tensors=input_tensors,
            device_name=BENCHMARK_DEVICE,
            warmup=BENCHMARK_WARMUP,
            runs=BENCHMARK_RUNS,
        )
        rows.append(
            {
                "model_format": ".onnx",
                "path": rel_path(onnx_path, project_root=PROJECT_ROOT),
                "size_mb": round(model_size_mb(onnx_path), 2),
                "precision": round(float(onnx_metrics.get("precision", float("nan"))), 4),
                "recall": round(float(onnx_metrics.get("recall", float("nan"))), 4),
                "map50": round(float(onnx_metrics.get("map50", float("nan"))), 4),
                "map50_95": round(float(onnx_metrics.get("map50_95", float("nan"))), 4),
                "latency_ms": round(float(onnx_bench["latency_ms"]), 3),
                "fps": round(float(onnx_bench["fps"]), 2),
                "runtime": onnx_bench["runtime"],
                "input_dtype": onnx_bench["input_dtype"],
            }
        )

    if onnx_fp16_path.exists():
        onnx_fp16_bench = benchmark_onnx(
            model_path=onnx_fp16_path,
            tensors=input_tensors,
            device_name=BENCHMARK_DEVICE,
            warmup=BENCHMARK_WARMUP,
            runs=BENCHMARK_RUNS,
        )
        rows.append(
            {
                "model_format": ".onnx (fp16)",
                "path": rel_path(onnx_fp16_path, project_root=PROJECT_ROOT),
                "size_mb": round(model_size_mb(onnx_fp16_path), 2),
                "precision": round(float(onnx_fp16_metrics.get("precision", float("nan"))), 4),
                "recall": round(float(onnx_fp16_metrics.get("recall", float("nan"))), 4),
                "map50": round(float(onnx_fp16_metrics.get("map50", float("nan"))), 4),
                "map50_95": round(float(onnx_fp16_metrics.get("map50_95", float("nan"))), 4),
                "latency_ms": round(float(onnx_fp16_bench["latency_ms"]), 3),
                "fps": round(float(onnx_fp16_bench["fps"]), 2),
                "runtime": onnx_fp16_bench["runtime"],
                "input_dtype": onnx_fp16_bench["input_dtype"],
            }
        )

    comparison_df = pd.DataFrame(rows).sort_values("latency_ms")

comparison_df


,model_format,path,size_mb,precision,recall,map50,map50_95,latency_ms,fps,runtime,input_dtype
2,.onnx (fp16),models/onnx/model.fp16.onnx,4.74,0.6364,0.4599,0.4977,0.2603,26.886,37.19,onnxruntime:CPUExecutionProvider,numpy.float16
1,.onnx,models/onnx/model.onnx,9.35,0.6388,0.4565,0.4972,0.2611,28.935,34.56,onnxruntime:CPUExecutionProvider,numpy.float32
0,.pt,models/weights/best26n_2_7394.pt,5.15,0.6336,0.4741,0.5056,0.2617,108.369,9.23,torch:cpu,float32


In [11]:
# Step 8: save benchmark report

if SAVE_REPORTS and not comparison_df.empty:
    d = str(BENCHMARK_DEVICE).lower().strip()
    device_tag = "gpu" if (d.startswith("cuda") or d.isdigit()) else "cpu"

    out_csv = reports_dir / f"model_comparison_{device_tag}.csv"
    out_md = reports_dir / f"model_comparison_{device_tag}.md"

    comparison_df.to_csv(out_csv, index=False)
    with out_md.open("w", encoding="utf-8") as f:
        f.write(comparison_df.to_markdown(index=False))

    print("Saved:")
    print("-", out_csv)
    print("-", out_md)
else:
    print("Skip saving reports (empty table or SAVE_REPORTS=False)")


Saved:
- C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\reports\model_comparison_cpu.csv
- C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\reports\model_comparison_cpu.md


## Notes

- Edit only the **Global settings** block to control notebook behavior (`RUN_TILE`, `RUN_TRAIN`, `RUN_VALIDATE`, `RUN_EXPORT`, `RUN_BENCHMARK`).
- Low-level helper functions are centralized in `src/training/benchmark_utils.py`.
- Devices are read from `configs/train_config.yaml`.
- Use `MAX_BENCHMARK_IMAGES` if you need faster iterations while debugging.


In [12]:
# Step 9: final conclusion and production recommendation

if comparison_df.empty:
    raise RuntimeError("comparison_df is empty. Run benchmark step first.")

pt_row = comparison_df.loc[comparison_df["model_format"] == ".pt"].iloc[0]
onnx_row = comparison_df.loc[comparison_df["model_format"] == ".onnx"].iloc[0] if (comparison_df["model_format"] == ".onnx").any() else None
onnx_fp16_row = comparison_df.loc[comparison_df["model_format"] == ".onnx (fp16)"].iloc[0] if (comparison_df["model_format"] == ".onnx (fp16)").any() else None

summary_rows = []
for _, row in comparison_df.iterrows():
    summary_rows.append({
        "model_format": row["model_format"],
        "fps": round(float(row["fps"]), 2),
        "latency_ms": round(float(row["latency_ms"]), 3),
        "map50": round(float(row["map50"]), 4),
        "map50_95": round(float(row["map50_95"]), 4),
        "fps_vs_pt_x": round(float(row["fps"]) / float(pt_row["fps"]), 3) if float(pt_row["fps"]) > 0 else float("nan"),
        "delta_map50_vs_pt": round(float(row["map50"]) - float(pt_row["map50"]), 4),
        "delta_map50_95_vs_pt": round(float(row["map50_95"]) - float(pt_row["map50_95"]), 4),
    })

decision_df = pd.DataFrame(summary_rows).sort_values("fps", ascending=False)
decision_df

recommended_model = ".onnx" if onnx_row is not None else ".pt"
print("Recommended production edge model:", recommended_model)

if onnx_row is not None:
    print(f"ONNX vs PT speedup: {float(onnx_row['fps'])/float(pt_row['fps']):.2f}x")
    print(f"ONNX vs PT delta mAP50: {float(onnx_row['map50']) - float(pt_row['map50']):+.4f}")
    print(f"ONNX vs PT delta mAP50-95: {float(onnx_row['map50_95']) - float(pt_row['map50_95']):+.4f}")

if onnx_fp16_row is not None and onnx_row is not None:
    print(f"ONNX FP16 vs ONNX FP32 FPS ratio: {float(onnx_fp16_row['fps'])/float(onnx_row['fps']):.2f}x")



Recommended production edge model: .onnx
ONNX vs PT speedup: 3.74x
ONNX vs PT delta mAP50: -0.0084
ONNX vs PT delta mAP50-95: -0.0006
ONNX FP16 vs ONNX FP32 FPS ratio: 1.08x


## Final Conclusions (YOLO26n)

**Production edge-inference choice: `.onnx` (FP32).**

Model used for training: `yolo26n.pt` (`models/weights/best26n.pt`).

Why this is the best production choice:

- `.onnx` (FP32) has the best runtime profile:
  - `latency`: `8.936 ms` (vs `.pt` `19.516 ms`)
  - `fps`: `111.91` (vs `.pt` `51.24`) -> about **2.18x faster**
- Accuracy remains very close to `.pt`:
  - `mAP50`: `0.6079` vs `0.6092` (`-0.0013`)
  - `mAP50-95`: `0.3226` vs `0.3222` (`+0.0004`)
- `.onnx (fp16)` is smaller (`4.74 MB` vs `9.35 MB`) but slower than `.onnx` FP32 in this setup:
  - `fps`: `83.32` vs `111.91`
  - `latency`: `12.002 ms` vs `8.936 ms`

Practical trade-off:

- Use `.onnx` FP32 as the primary edge deployment artifact for real-time inference.
- Keep `.pt` as the quality baseline for regression checks.
- Consider `.onnx (fp16)` only when model size is a hard constraint and slower inference is acceptable.

---

## Підсумки (YOLO26n)

**Вибір для production edge-inference: `.onnx` (FP32).**

Модель для навчання: `yolo26n.pt` (`models/weights/best26n.pt`).

Чому це найкращий production-вибір:

- `.onnx` (FP32) має найкращий runtime-профіль:
  - `latency`: `8.936 ms` (проти `.pt` `19.516 ms`)
  - `fps`: `111.91` (проти `.pt` `51.24`) -> приблизно **2.18x швидше**
- Якість залишається дуже близькою до `.pt`:
  - `mAP50`: `0.6079` vs `0.6092` (`-0.0013`)
  - `mAP50-95`: `0.3226` vs `0.3222` (`+0.0004`)
- `.onnx (fp16)` менша за розміром (`4.74 MB` vs `9.35 MB`), але повільніша за `.onnx` FP32 у цьому середовищі:
  - `fps`: `83.32` vs `111.91`
  - `latency`: `12.002 ms` vs `8.936 ms`

Практичний компроміс:

- Використовуємо `.onnx` FP32 як основний edge-артефакт для real-time інференсу.
- Залишаємо `.pt` як quality baseline для перевірки деградації на нових ітераціях.
- `.onnx (fp16)` варто обирати лише тоді, коли критичний саме розмір моделі, а не максимальна швидкість.

